In [3]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn  as sns

In [5]:
df=pd.read_csv('retail_store_inventory.csv')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  object 
 1   Store ID            73100 non-null  object 
 2   Product ID          73100 non-null  object 
 3   Category            73100 non-null  object 
 4   Region              73100 non-null  object 
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  object 
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  object 
dtypes: float64(3), int64(5), object(7)
memory usage: 8.4+

In [7]:
# Convert 'Date' column to datetime objects
df['Date'] = pd.to_datetime(df['Date'])

# Sort the data by Date
df = df.sort_values(by='Date')

# Encode categorical variables (e.g., Category, Region, Weather, Seasonality)
# For simplicity, we can use pandas get_dummies (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=['Category', 'Region', 'Weather Condition', 'Seasonality'], drop_first=True)


In [8]:
# 1. Time Features
df_encoded['DayOfWeek'] = df_encoded['Date'].dt.dayofweek
df_encoded['Month'] = df_encoded['Date'].dt.month
df_encoded['Day'] = df_encoded['Date'].dt.day

# 2. Lag Features (e.g., Units Sold 1 day ago, 7 days ago)
# Note: In a real scenario, you'd calculate lags per Store and Product
df_encoded['Units_Sold_Lag_1'] = df_encoded.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(1)
df_encoded['Units_Sold_Lag_7'] = df_encoded.groupby(['Store ID', 'Product ID'])['Units Sold'].shift(7)

# Drop rows with NaN values created by lagging
df_encoded = df_encoded.dropna() 


In [9]:
# Let's say we use the last 20% of dates as our test set
split_date = df_encoded['Date'].quantile(0.8)

train = df_encoded[df_encoded['Date'] < split_date]
test = df_encoded[df_encoded['Date'] >= split_date]

# Define features (X) and Target (y) - Target is 'Units Sold'
drop_cols = ['Date', 'Store ID', 'Product ID', 'Units Sold', 'Demand Forecast']
X_train = train.drop(columns=drop_cols)
y_train = train['Units Sold']

X_test = test.drop(columns=drop_cols)
y_test = test['Units Sold']


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Initialize the model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
predictions = model.predict(X_test)


In [ ]:
# Calculate Errors
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

#Plot Actual vs Predicted for a quick visual check
plt.figure(figsize=(10,5))
plt.plot(y_test.values[:100], label='Actual Sales')
plt.plot(predictions[:100], label='Predicted Sales')
plt.legend()
plt.title('Forecasting: Actual vs Predicted')
plt.show()
